In [1]:
 !pip install -q outlines transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00


In [2]:
from pydantic import BaseModel, Field
from typing import List

class EliminatedScenario(BaseModel):
    scenario: str = Field(description="The ruled-out explanation")
    reason: str = Field(description="Why it's ruled out")




class Schema(BaseModel):

    problem: str = Field(
        description="The prompt content — can be anything from a short piece of information ,a news snippet, to a fully explained, detail-rich scenario. No fixed length or format required; "
    )
    observations: List[str] = Field(
        description="A list of observations made from the problem statement"
    )
    possible_scenario: List[str] = Field(
        description="A list of plausible explanations or hypotheses that could account for the clue in `problem`. Each item should be a distinct, concrete possibility — not a rephrasing of another item."
    )
    impossible_scenario: List[EliminatedScenario]

    deduction: str = Field(
        description="The connective reasoning in Holmes's voice: walks from `problem`, through why the ruled_out options in `analysing_possibilities` fail, to why the surviving option(s) fit. Explain the 'why', don't just restate the options."
    )
    final_answer: str = Field(
        description="The concise final verdict reached at the end of `deduction` — one to two sentences stating the conclusion directly, with no new reasoning introduced here."
    )



In [3]:
"""
llm.py — local Qwen2.5 replacement for the Groq/Gemini/NVIDIA SmartLLM chain.

Same public interface as before:
    llm = get_llm(output_schema=Schema)
    result = llm.invoke(prompt)   # -> a Schema instance, so result.model_dump() still works

Nothing in main generation loop needs to change.



Colab setup (run once in a cell — do NOT add torch to this line, see earlier notes):
    !pip install -q outlines transformers accelerate bitsandbytes

GPU notes:
  - Qwen2.5-7B-Instruct needs ~15GB just for bf16 weights. On a T4 (16GB total)
    that leaves almost nothing for activations/KV cache
  - Default here is 4-bit quantization (load_in_4bit), which comfortably fits
    7B on a T4 with room to spare. On an A100 you can turn that off and load
    in bf16 for better quality/speed — flip USE_4BIT below.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import outlines

# ── Config ─────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS = 1500
USE_4BIT = True   # set False if you're on an A100 and want full bf16 quality/speed

_model = None
_tokenizer = None
_outlines_model = None


def _load_model():
    global _model, _tokenizer, _outlines_model
    if _outlines_model is not None:
        return _outlines_model

    print(f"Loading {MODEL_NAME} (4-bit={USE_4BIT}) ...")
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    load_kwargs = {"device_map": "auto"}
    if USE_4BIT:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )
    else:
        load_kwargs["dtype"] = torch.bfloat16

    _model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    _outlines_model = outlines.from_transformers(_model, _tokenizer)
    print("Model loaded.")
    return _outlines_model


class LocalQwenLLM:
    """
    Drop-in replacement for SmartLLM. Runs generation locally and uses
    outlines' constrained JSON decoding, so the model is forced to emit
    something that matches `output_schema` on every call — no fallback
    chain needed since there's only one model.
    """
    def __init__(self, output_schema):
        self.model = _load_model()
        self.tokenizer = _tokenizer
        self.output_schema = output_schema
        # Building the Generator once (not per-call) matters: it compiles the
        # schema's constraint machinery, which is expensive to redo every sample.
        self.generator = outlines.Generator(self.model, output_schema)

    def _format(self, prompt: str) -> str:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a meticulous dataset-generation assistant. "
                    "Respond with a single JSON object matching the requested schema only."
                ),
            },
            {"role": "user", "content": prompt},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _parse(self, raw):
        # outlines v1 generators return a JSON string, not a validated Pydantic
        # instance -- validate it ourselves so .model_dump() downstream still works.
        if isinstance(raw, self.output_schema):
            return raw
        return self.output_schema.model_validate_json(raw)

    def invoke(self, prompt: str):
        formatted = self._format(prompt)
        raw = self.generator(formatted, max_new_tokens=MAX_NEW_TOKENS)
        return self._parse(raw)


def get_llm(output_schema=None):
    if output_schema is None:
        raise ValueError("This local pipeline requires a structured output_schema (Pydantic model).")
    return LocalQwenLLM(output_schema)


# ── Example usage ──────────────────────────────────────────────────
if __name__ == "__main__":
    llm = get_llm(output_schema=Schema)
    result = llm.invoke("Generate one reasoning-style dataset sample in the given schema.")
    print(result.model_dump())

Loading Qwen/Qwen2.5-7B-Instruct (4-bit=True) ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded.
{'problem': 'If a car travels 150 miles using 5 gallons of fuel, what is its fuel efficiency in miles per gallon?', 'observations': ['The car travels 150 miles', 'The car uses 5 gallons of fuel'], 'possible_scenario': ['We can calculate the fuel efficiency by dividing the distance traveled by the amount of fuel used.'], 'impossible_scenario': [], 'deduction': "Therefore, the car's fuel efficiency is 30 miles per gallon (150 miles / 5 gallons = 30 miles per gallon).", 'final_answer': "The car's fuel efficiency is 30 miles per gallon."}


In [4]:

llm = get_llm(output_schema=Schema)
context='everyday logical reasoning'
difficulty=3
prompt=prompt = f"""Generate one reasoning-style dataset sample.

                Domain: {context}
                Target difficulty: {difficulty} (1=single obvious inference, 5=multiple interacting clues with several plausible-looking traps)

                Rules:
                    - Every fact used in `observations`, `impossible_scenario` reasons, and `deduction` MUST come from `problem` or from a fact already stated in `observations`.
                    - Do NOT assert the absence, non-occurrence, or non-existence of something (e.g. "no alarm was triggered," "no delivery was scheduled," "records show nothing," "no footprints were found") unless a prior `observations` entry explicitly states that the relevant check, log review, or witness canvass was performed. If an absence is needed for the deduction, first add an observation stating the check happened, before using its result.
                    - Do not introduce new named entities, events, dates, or documents that are not present in `problem`.
                """
result = llm.invoke(prompt)
print(result.model_dump())

{'problem': "A company's server was down for maintenance on Tuesday. The IT manager, John, checked the server logs and found that a backup had been initiated at 9 AM. However, the system administrator, Sarah, claims she did not start any backups that day.", 'observations': ['John checked the server logs and found a backup initiation at 9 AM on Tuesday.', 'Sarah claims she did not start any backups on Tuesday.'], 'possible_scenario': ['Sarah might have forgotten to start the backup, but it was automatically initiated by a scheduled task.', "The backup could have been started by another team member without Sarah's knowledge."], 'impossible_scenario': [{'scenario': 'The backup was manually started by Sarah after John checked the logs.', 'reason': "This contradicts Sarah's claim that she did not start any backups on Tuesday."}, {'scenario': 'There was no backup initiated on Tuesday.', 'reason': 'The logs clearly show a backup initiation at 9 AM on Tuesday.'}], 'deduction': "Given the evide

In [ ]:

import json
import os

contexts = ['crime/mystery scenario', 'everyday logical reasoning', 'scientific reasoning', 'pure information/facts']
llm = get_llm(output_schema=Schema)

OUTPUT_PATH = "/content/drive/MyDrive/dataset_for_SMOLLM/dataset_1.jsonl"
TOTAL_LOOPS = 50
EXAMPLES_PER_LOOP = len(contexts) * 5   # 4 contexts * 5 difficulties = 20
TOTAL_TARGET = TOTAL_LOOPS * EXAMPLES_PER_LOOP

def get_line_count(path):
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

existing_count = get_line_count(OUTPUT_PATH)
start_loop = existing_count // EXAMPLES_PER_LOOP
print(f"Found {existing_count} existing examples. Resuming from loop {start_loop}/{TOTAL_LOOPS}")
print(f"Target: {TOTAL_TARGET} total examples")

success_count = 0
fail_count = 0

with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
    for loop in range(start_loop,TOTAL_LOOPS):
        for context in contexts:
            for difficulty in range(1,6):
                prompt = f"""Generate one reasoning-style dataset sample.

                Domain: {context}
                Target difficulty: {difficulty} (1=single obvious inference, 5=multiple interacting clues with several plausible-looking traps)

                Rules:
                    - Every fact used in `observations` MUST come from from a fact already stated in `problem` or known facts, don't include any assumptions.
                    - Every fact used in `observations`, `impossible_scenario` reasons, and `deduction` MUST come from `problem` or from a fact already stated in `observations`.
                    - Do NOT assert the absence, non-occurrence, or non-existence of something (e.g. "no alarm was triggered," "no delivery was scheduled," "records show nothing," "no footprints were found") unless a prior `observations` entry explicitly states that the relevant check, log review, or witness canvass was performed. If an absence is needed for the deduction, first add an observation stating the check happened, before using its result.
                    - Do not introduce new named entities, events, dates, or documents that are not present in `problem`.
                """
                try:
                    result = llm.invoke(prompt)
                    record = result.model_dump()   # pydantic model -> dict
                    record["context"] = context
                    record["difficulty"] = difficulty
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()  # write immediately so you don't lose progress on a crash
                    success_count += 1

                except Exception as e:
                    fail_count += 1
                    print(f"[loop {loop}] [{context}] FAILED: {e}")
                    continue  # skip this sample, move to the next
                total_written = existing_count + success_count
                remaining = TOTAL_TARGET - total_written
                print(f"Generated: {total_written}/{TOTAL_TARGET} | Remaining: {remaining} | fails: {fail_count}")

print(f"Finished. success={success_count} fail={fail_count}")


# print(result.problem)
# print(result.observations)
# print(result.possible_scenario)
# print(result.impossible_scenario)
# print(result.deduction)
# print(result.final_answer)
# print(result.difficulty)


Found 0 existing examples. Resuming from loop 0/50
Target: 1000 total examples
Generated: 1/1000 | Remaining: 999 | fails: 0
Generated: 2/1000 | Remaining: 998 | fails: 0
Generated: 3/1000 | Remaining: 997 | fails: 0
Generated: 4/1000 | Remaining: 996 | fails: 0
Generated: 5/1000 | Remaining: 995 | fails: 0
Generated: 6/1000 | Remaining: 994 | fails: 0
Generated: 7/1000 | Remaining: 993 | fails: 0
Generated: 8/1000 | Remaining: 992 | fails: 0
Generated: 9/1000 | Remaining: 991 | fails: 0
Generated: 10/1000 | Remaining: 990 | fails: 0
Generated: 11/1000 | Remaining: 989 | fails: 0
Generated: 12/1000 | Remaining: 988 | fails: 0
Generated: 13/1000 | Remaining: 987 | fails: 0
Generated: 14/1000 | Remaining: 986 | fails: 0
Generated: 15/1000 | Remaining: 985 | fails: 0
Generated: 16/1000 | Remaining: 984 | fails: 0
Generated: 17/1000 | Remaining: 983 | fails: 0
Generated: 18/1000 | Remaining: 982 | fails: 0
Generated: 19/1000 | Remaining: 981 | fails: 0
Generated: 20/1000 | Remaining: 980 |